# Export the v2 model for the browser

Turns `modele_hdfs_v2` (LoRA adapter) into a 4-bit ONNX model that runs in the browser with WebGPU, and uploads it to Hugging Face, where `docs/index.html` loads it from.

A CPU runtime is enough (no GPU needed). It takes about 20-30 minutes, mostly the export. It needs a lot of RAM (the model is exported in fp32, about 6 GB): if the Colab session crashes for lack of memory, run the same notebook on Kaggle (30 GB of RAM for free) or with a high-RAM runtime.

You need a Hugging Face account and a token with **write** access (huggingface.co > Settings > Access Tokens).

In [ ]:
!pip install -q "transformers==4.46.3" accelerate huggingface_hub safetensors "optimum-onnx[onnxruntime]" onnx onnx-ir onnxruntime
!git clone -q --depth 1 https://github.com/NafiyTP/RootCause-SLM.git /content/RootCause-SLM
!ls /content/RootCause-SLM/modele_hdfs_v2

## 1. Merge the LoRA adapter into the base model

Done by hand (W += alpha/r · B·A for each adapted matrix) instead of with peft, so it does not depend on the peft and torch versions installed on the machine.

In [ ]:
import json, torch
from safetensors.torch import load_file
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER = "/content/RootCause-SLM/modele_hdfs_v2"
cfg = json.load(open(f"{ADAPTER}/adapter_config.json"))
assert not cfg.get("use_dora") and not cfg.get("modules_to_save")
scale = cfg["lora_alpha"] / (cfg["r"] ** 0.5 if cfg.get("use_rslora") else cfg["r"])

model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float32)
params = dict(model.named_parameters())
lora = load_file(f"{ADAPTER}/adapter_model.safetensors")
n = 0
with torch.no_grad():
    for k, A in lora.items():
        if ".lora_A." not in k:
            continue
        B = lora[k.replace(".lora_A.", ".lora_B.")]
        name = k.replace("base_model.model.", "", 1).replace(".lora_A.weight", ".weight")
        params[name] += scale * (B.float() @ A.float())
        n += 1
print(n, "matrices merged, scale", scale)
assert n == len(cfg["target_modules"]) * model.config.num_hidden_layers
model.save_pretrained("/content/merged", safe_serialization=True)
AutoTokenizer.from_pretrained(BASE).save_pretrained("/content/merged")
del model, params, lora
import gc; gc.collect()
!ls -la /content/merged

## 2. Export to ONNX (with KV cache, for fast generation)

In [ ]:
!optimum-cli export onnx --model /content/merged --task text-generation-with-past /content/onnx_fp32
!rm -rf /content/merged
!ls -la /content/onnx_fp32

## 3. Quantize the weights to 4 bits

`onnx/model_q4.onnx` + `onnx/model_q4.onnx_data` (4-bit weights, the embedding stays fp32, about 1.7 GB in total) is what the page loads. The weights are always saved in a separate `_data` file, which is what the browser runtime handles best for a model this size. The fp32 export (about 6 GB) is not uploaded.

In [ ]:
import os, shutil, glob, json, gc, onnx
from onnxruntime.quantization.matmul_nbits_quantizer import MatMulNBitsQuantizer

WEB = "/content/web"
os.makedirs(f"{WEB}/onnx", exist_ok=True)
m = onnx.load("/content/onnx_fp32/model.onnx")          # also loads model.onnx_data
q = MatMulNBitsQuantizer(m, block_size=32, is_symmetric=True)
q.process()
del m; gc.collect()
onnx.save_model(q.model.model, f"{WEB}/onnx/model_q4.onnx", save_as_external_data=True,
                all_tensors_to_one_file=True, location="model_q4.onnx_data", size_threshold=0)
del q; gc.collect()
for f in glob.glob("/content/onnx_fp32/*.json") + glob.glob("/content/onnx_fp32/*.txt"):
    shutil.copy(f, WEB)                                     # config, tokenizer, generation config
cfg = json.load(open(f"{WEB}/config.json"))
cfg["transformers.js_config"] = {"dtype": "q4", "use_external_data_format": True}
json.dump(cfg, open(f"{WEB}/config.json", "w"), indent=2)
!ls -la /content/web /content/web/onnx

## 4. Quick check on CPU

The answer should be a JSON with `anomalous`, `cause`, `evidence`, `explanation`.

In [ ]:
import json, sys
sys.path.insert(0, "/content/RootCause-SLM/src")
from blocks_v2 import SYSTEM_PROMPT
from transformers import AutoTokenizer
from optimum.onnxruntime import ORTModelForCausalLM
tok = AutoTokenizer.from_pretrained(WEB)
ort = ORTModelForCausalLM.from_pretrained(WEB, subfolder="onnx", file_name="model_q4.onnx")
x = json.load(open("/content/RootCause-SLM/data/v2/test_blocks.json"))[0]
msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": x["prompt"]}]
ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt")
out = ort.generate(ids, max_new_tokens=200, do_sample=False)
print(x["category"], "->", tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True))

## 5. Upload to Hugging Face

The repo is public (the page downloads it from the visitor's browser).

In [ ]:
import getpass
from huggingface_hub import HfApi
api = HfApi(token=getpass.getpass("Hugging Face token (write): "))
user = api.whoami()["name"]
REPO_ID = f"{user}/rootcause-slm-v2-web"
api.create_repo(REPO_ID, exist_ok=True)
api.upload_folder(folder_path=WEB, repo_id=REPO_ID, commit_message="v2 model, ONNX 4-bit for the browser")
print("uploaded to https://huggingface.co/" + REPO_ID)
if REPO_ID != "NafiyTP/rootcause-slm-v2-web":
    print("in docs/index.html, change MODEL_ID to:", REPO_ID)